# 🫁 Enhanced Cascade Pipeline v4 — Complete Edition
## Architecture
```
Input X-Ray
      │
      ▼
┌──────────────────────────────────────────────────────┐
│  SHARED PREPROCESSING  (all images)                  │
│  Gaussian Blur → Percentile Norm → CLAHE             │
│  Lobar ROI Masking (Otsu + morphological fill)       │
│  Periphery Blurring                                  │
└──────────────────────────────────────────────────────┘
      │
      ▼
┌─────────────────────┐
│  Stage 1: ViT-B/16  │  →  NORMAL  or  PNEUMONIA
└─────────────────────┘
      │              │
   NORMAL        PNEUMONIA
      │      ┌───────┴────────┐  (both run simultaneously)
      │      │                │
      │  ┌───┴──────────┐  ┌──┴──────────────┐
      │  │ Stage 3       │  │  Stage 2         │
      │  │ BACTERIAL CNN │  │  VIRAL CNN       │
      │  │ Sobel Edge    │  │  Bilateral Filt  │
      │  │ Unsharp Mask  │  │  Morph Dilation  │
      │  │ High-Pass     │  │  Hist Matching   │
      │  │ LobarROI      │  │  Median Blur ROI │
      │  │ PeriBlur      │  │  Aniso DiffNorm  │
      │  └───────────────┘  └──────────────────┘
      │           │                   │
      │    ┌──────┴───────────────────┴──────┐
      │    │   Decision Logic + DI Fusion     │
      │    │  Viral=YES  Bact=YES → CO-INF   │
      │    │  Viral=YES  Bact=NO  → VIRAL    │
      │    │  Viral=NO   Bact=YES → BACT     │
      │    │  Viral=NO   Bact=NO  → AMBIGUOUS│
      │    └─────────────────────────────────┘
      │                AMBIGUOUS
      │                   │
      │         attempt 1 → 🔁 Resend Stage 1
      │         attempt 2 → ⚠️  MISCELLANEOUS
   ✅ NORMAL

──────────────────────────────────────────────────────
🧠 Density Index (DI):  DI = mean_lung / std_texture
   25% soft-fusion auxiliary score
🔁 5-Fold Stratified Cross-Validation on all 3 stages
🎛️  Dataset Size Controller — one cell to change all sizes
💾 Model Persistence — auto-saves to /kaggle/working/saved_models/
──────────────────────────────────────────────────────
Labels: 0=NORMAL | 1=VIRAL | 2=BACTERIAL | 3=CO-INFECTION | 4=MISCELLANEOUS
```

In [ ]:

import subprocess, torch

print(f"Current PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    cap  = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print(f"GPU             : {name}")
    print(f"Compute cap     : sm_{cap[0]}{cap[1]}")
else:
    print("No CUDA detected — checking nvidia-smi...")

result = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap",
                         "--format=csv,noheader"], capture_output=True, text=True)
print(f"nvidia-smi      : {result.stdout.strip()}")

# Auto-reinstall correct cu121 wheels for sm_80+
cuda_cap = None
if torch.cuda.is_available():
    cuda_cap = torch.cuda.get_device_capability(0)
else:
    try:
        cap_str = result.stdout.strip().split(",")[-1].strip().replace(".", "")
        cuda_cap = (int(cap_str[0]), int(cap_str[1]))
    except:
        pass

print(f"\nDetected compute capability: {cuda_cap}")

if cuda_cap and cuda_cap[0] >= 8 and "cu121" not in torch.__version__:
    wheel = "https://download.pytorch.org/whl/cu121"
    print(f"→ Reinstalling PyTorch cu121 for sm_{cuda_cap[0]}{cuda_cap[1]}...")
    subprocess.run([
        "pip", "install", "-q", "--upgrade",
        "torch==2.3.0", "torchvision==0.18.0",
        "--index-url", wheel
    ], check=True)
    print("✅ Done — RESTART KERNEL, then run from Cell 2 onwards.")
else:
    print("✅ PyTorch version looks fine — no reinstall needed.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2: IMPORTS & DEVICE
# ═══════════════════════════════════════════════════════════════════
import os, shutil, random, glob, copy
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import vit_b_16, ViT_B_16_Weights
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    f1_score, accuracy_score, precision_score, recall_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.ndimage import gaussian_filter
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

ImageFile.LOAD_TRUNCATED_IMAGES = True

if torch.cuda.is_available():
    try:
        _t = torch.zeros(1, device="cuda") + 1
        del _t
        print(f"✅ GPU OK : {torch.cuda.get_device_name(0)}")
        print(f"   Compute cap  : sm_{torch.cuda.get_device_capability(0)}")
        print(f"   PyTorch CUDA : {torch.version.cuda}")
        device = torch.device("cuda")
    except Exception as e:
        print(f"❌ GPU failing: {e}  → Re-run Cell 1, restart kernel")
        device = torch.device("cpu")
else:
    device = torch.device("cpu")
    print("⚠️  No CUDA — using CPU")

print(f"🚀 Device  : {device}")
print(f"PyTorch   : {torch.__version__}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"🌱 Seed set → {seed}")

set_seed(42)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3: PATHS
# ═══════════════════════════════════════════════════════════════════
# ── Input dataset (chest-xray-pneumonia ImageFolder structure) ────
DATASET_PATH = '/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray/'
TRAIN_PATH   = os.path.join(DATASET_PATH, 'train')
VAL_PATH     = os.path.join(DATASET_PATH, 'val')
TEST_PATH    = os.path.join(DATASET_PATH, 'test')

# ── Working directories for reorganised sub-class folders ─────────
VIRAL_WORK = '/kaggle/working/pneumofusion/'
BACT_WORK  = '/kaggle/working/bacterialmodel/'

# ── Checkpoint paths ──────────────────────────────────────────────
CKPT_VIT   = '/kaggle/working/stage1_vit_best.pth'
CKPT_VIRAL = '/kaggle/working/stage2_viral_best.pth'
CKPT_BACT  = '/kaggle/working/stage3_bact_best.pth'

# ── Persistent model save dir (survives session) ──────────────────
SAVED_MODELS_DIR = '/kaggle/working/saved_models/'
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

print('✅ Paths configured')
for name, p in [('Train/NORMAL',    os.path.join(TRAIN_PATH,'NORMAL')),
                ('Train/PNEUMONIA', os.path.join(TRAIN_PATH,'PNEUMONIA')),
                ('Val/NORMAL',      os.path.join(VAL_PATH,  'NORMAL')),
                ('Val/PNEUMONIA',   os.path.join(VAL_PATH,  'PNEUMONIA')),
                ('Test/NORMAL',     os.path.join(TEST_PATH, 'NORMAL')),
                ('Test/PNEUMONIA',  os.path.join(TEST_PATH, 'PNEUMONIA'))]:
    exists = os.path.exists(p)
    count  = len(os.listdir(p)) if exists else 0
    print(f'   {name:<22}: {"✅" if exists else "❌"}  ({count} files)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4: 🎛️  DATASET SIZE CONTROLLER  ← CHANGE NUMBERS HERE ONLY
# ═══════════════════════════════════════════════════════════════════
#
#  Set any value to None to use the FULL split.
#
#  ── Presets ──────────────────────────────────────────────────────
#    CPU quick test  : TRAIN=500,  VAL=100,  TEST=200
#    CPU moderate    : TRAIN=2000, VAL=400,  TEST=500
#    GPU full run    : set ALL to None
#  ─────────────────────────────────────────────────────────────────

TRAIN_SUBSET = 100    # ← images per train split  (None = full ~5216)
VAL_SUBSET   = 2   # ← images per val split    (None = full ~16)
TEST_SUBSET  = 10    # ← images per test split   (None = full ~624)

# ── Training epoch controls ───────────────────────────────────────
STAGE1_EPOCHS = 1     # ViT  (reduce to 5 for CPU)
STAGE2_EPOCHS = 1     # Viral CNN
STAGE3_EPOCHS = 1     # Bacterial CNN
KFOLD_EPOCHS  = 1     # epochs per K-Fold fold
N_FOLDS       = 1      # number of cross-validation folds

# ─────────────────────────────────────────────────────────────────
print("🎛️  Dataset size settings:")
print(f"   Train  : {TRAIN_SUBSET if TRAIN_SUBSET else 'FULL'}")
print(f"   Val    : {VAL_SUBSET   if VAL_SUBSET   else 'FULL'}")
print(f"   Test   : {TEST_SUBSET  if TEST_SUBSET  else 'FULL'}")
print(f"   Epochs : S1={STAGE1_EPOCHS}  S2={STAGE2_EPOCHS}  S3={STAGE3_EPOCHS}  KFold={KFOLD_EPOCHS}×{N_FOLDS}")


def sample_folder_dataset(dataset, n, seed=42):
    """
    Stratified subsample of an ImageFolder/FolderDataset,
    preserving class balance. Returns a list of (path, label) tuples.
    n=None returns all samples.
    """
    if n is None or n >= len(dataset):
        return list(zip(dataset.samples if hasattr(dataset,'samples')
                        else dataset.image_paths,
                        dataset.labels))
    labels = np.array(dataset.labels)
    classes, counts = np.unique(labels, return_counts=True)
    rng = np.random.RandomState(seed)
    selected = []
    for cls, cnt in zip(classes, counts):
        idx = np.where(labels == cls)[0]
        k   = max(1, int(round((cnt / len(labels)) * n)))
        k   = min(k, len(idx))
        chosen = rng.choice(idx, k, replace=False)
        if hasattr(dataset, 'samples'):
            selected += [dataset.samples[i] for i in chosen]
        else:
            selected += [(dataset.image_paths[i], dataset.labels[i])
                         for i in chosen]
    rng.shuffle(selected)
    return selected[:n]


print("\n✅ sample_folder_dataset() ready — stratified, class-balanced")
print("   Set TRAIN_SUBSET = None to use every available image.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5: ADVANCED PREPROCESSING CLASSES
#   Base:       Gaussian Blur + Percentile Norm + CLAHE
#   LobarROI:   Otsu → morphological fill → safety ellipse fallback
#   Bacterial:  Sobel Edge + Unsharp Mask + High-Pass + PeriBlur
#   Viral:      Bilateral + MorphDil + HistMatch + MedianROI
#               + Anisotropic Diffusion Norm + PeriBlur
# ═══════════════════════════════════════════════════════════════════

def to_uint8(img_float):
    """Clip [0,1] → uint8 safely."""    return (np.clip(img_float, 0.0, 1.0) * 255).astype(np.uint8)

def to_float(img_uint8):
    return img_uint8.astype(np.float32) / 255.0


# ── 5-A  BASE PREPROCESSOR ───────────────────────────────────────
class BaseChestPreprocessor:
    """Gaussian Blur → Percentile Norm → CLAHE → RGB PIL"""    def __init__(self, clahe_clip=3.0, tile=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=tile)

    def to_gray(self, pil_img):
        img = np.array(pil_img)
        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        return img  # uint8

    def base_pipeline(self, pil_img):
        """Returns float32 H×W in [0,1] after blur+percnorm+CLAHE."""        gray = self.to_gray(pil_img)
        gray = cv2.GaussianBlur(gray, (3, 3), 0)
        p2, p98 = np.percentile(gray, (2, 98))
        gray = np.clip(gray.astype(np.float32), p2, p98)
        gray = (gray - p2) / (p98 - p2 + 1e-8)
        gray = self.clahe.apply(to_uint8(gray))
        return to_float(gray)

    def to_rgb_pil(self, img_float):
        arr = to_uint8(img_float)
        return Image.fromarray(np.stack([arr, arr, arr], axis=-1))


# ── 5-B  LOBAR ROI MASKING ────────────────────────────────────────
class LobarROIMask:
    """
    Filled lung-field mask from RAW uint8 grayscale (pre-CLAHE).
    Returns float32 mask [0,1] same size as input.
    """
    @staticmethod
    def compute(raw_gray_uint8):
        h, w = raw_gray_uint8.shape
        blurred = cv2.GaussianBlur(raw_gray_uint8, (5, 5), 0)
        _, thresh = cv2.threshold(blurred, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        thresh = cv2.bitwise_not(thresh)
        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))
        mask = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, k_close, iterations=4)
        k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (10, 10))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k_open, iterations=2)

        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
        if n_labels >= 2:
            areas   = stats[1:, cv2.CC_STAT_AREA]
            n_keep  = min(2, len(areas))
            top_idx = np.argsort(areas)[-n_keep:] + 1
            clean   = np.zeros_like(mask)
            for lbl in top_idx:
                clean[labels == lbl] = 255
            k_fill = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (30, 30))
            clean  = cv2.dilate(clean, k_fill, iterations=3)
            clean  = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, k_fill, iterations=4)
            mask   = clean

        # Safety fallback: if mask < 10% of image, use central ellipse
        if mask.mean() / 255.0 < 0.10:
            mask   = np.zeros((h, w), dtype=np.uint8)
            cy, cx = h // 2, w // 2
            cv2.ellipse(mask, (cx, cy), (int(w*0.35), int(h*0.30)),
                        0, 0, 360, 255, -1)

        return mask.astype(np.float32) / 255.0

    @staticmethod
    def apply_roi(img_float, mask):
        return img_float * mask


# ── 5-C  PERIPHERY BLURRING ──────────────────────────────────────
class PeripheryBlur:
    def __init__(self, falloff=0.50, min_weight=0.20, blur_ksize=21, blur_sigma=15):
        self.falloff    = falloff
        self.min_weight = min_weight
        self.ks         = blur_ksize | 1
        self.blur_sigma = blur_sigma

    def __call__(self, img_float):
        h, w = img_float.shape
        cy, cx = h / 2.0, w / 2.0
        Y, X   = np.ogrid[:h, :w]
        dist   = np.sqrt((X - cx)**2 + (Y - cy)**2)
        sigma_px = max(h, w) * self.falloff
        weight   = np.exp(-dist**2 / (2 * sigma_px**2)).astype(np.float32)
        weight   = np.clip(weight, self.min_weight, 1.0)
        blurred  = cv2.GaussianBlur(img_float, (self.ks, self.ks), self.blur_sigma)
        return np.clip(weight * img_float + (1.0 - weight) * blurred, 0.0, 1.0)


# ── 5-D  BACTERIAL PREPROCESSOR ──────────────────────────────────
class BacterialPreprocessor(BaseChestPreprocessor):
    """
    Pipeline:  Raw → LobarROI mask
    CLAHE base → apply ROI
    → Sobel Edge Enhancement (blend=0.25)
    → Unsharp Masking (strength=1.2, σ=1.5)
    → High-Pass Filter (σ=8, α=0.35)
    → Periphery Blur → RGB PIL
    """
    def __init__(self):
        super().__init__(clahe_clip=2.5, tile=(8, 8))
        self.pblur = PeripheryBlur(falloff=0.50, min_weight=0.20,
                                   blur_ksize=21, blur_sigma=15)

    def sobel_enhance(self, img_float, blend=0.25):
        u8  = to_uint8(img_float)
        gx  = cv2.Sobel(u8, cv2.CV_64F, 1, 0, ksize=3)
        gy  = cv2.Sobel(u8, cv2.CV_64F, 0, 1, ksize=3)
        mag = np.sqrt(gx**2 + gy**2)
        mag = (mag / (mag.max() + 1e-8)).astype(np.float32)
        return np.clip(img_float + blend * mag, 0.0, 1.0)

    def unsharp_mask(self, img_float, strength=1.2, sigma=1.5):
        from scipy.ndimage import gaussian_filter
        blur  = gaussian_filter(img_float, sigma=sigma)
        return np.clip(img_float + strength * (img_float - blur), 0.0, 1.0)

    def high_pass_filter(self, img_float, sigma=8.0, alpha=0.35):
        from scipy.ndimage import gaussian_filter
        low    = gaussian_filter(img_float, sigma=sigma)
        hp     = img_float - low
        hp_norm = (hp - hp.min()) / (hp.max() - hp.min() + 1e-8)
        return np.clip(img_float + alpha * hp_norm, 0.0, 1.0)

    def __call__(self, pil_img):
        raw  = self.to_gray(pil_img)
        mask = LobarROIMask.compute(raw)
        img  = self.base_pipeline(pil_img)
        img  = LobarROIMask.apply_roi(img, mask)
        img  = self.sobel_enhance(img, blend=0.25)
        img  = self.unsharp_mask(img, strength=1.2, sigma=1.5)
        img  = self.high_pass_filter(img, sigma=8.0, alpha=0.35)
        img  = self.pblur(img)
        return self.to_rgb_pil(img)


# ── 5-E  VIRAL PREPROCESSOR ──────────────────────────────────────
class ViralPreprocessor(BaseChestPreprocessor):
    """
    Pipeline:  Raw → LobarROI mask
    CLAHE base → apply ROI
    → Bilateral Filter (edge-preserving)
    → Morphological Dilation (expand infiltrates)
    → Histogram Matching (standardise contrast)
    → Median Blur on ROI (speckle removal)
    → Anisotropic Diffusion Norm (Perona-Malik)
    → Periphery Blur → RGB PIL
    """
    def __init__(self):
        super().__init__(clahe_clip=3.5, tile=(8, 8))
        self.pblur = PeripheryBlur(falloff=0.50, min_weight=0.20,
                                   blur_ksize=25, blur_sigma=20)

    def bilateral_filter(self, img_float):
        u8 = to_uint8(img_float)
        f  = cv2.bilateralFilter(u8, d=9, sigmaColor=75, sigmaSpace=75)
        return to_float(f)

    def morphological_dilation(self, img_float, ksize=3):
        u8     = to_uint8(img_float)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
        return to_float(cv2.dilate(u8, kernel, iterations=1))

    def histogram_matching(self, img_float):
        u8   = to_uint8(img_float)
        hist, _ = np.histogram(u8.flatten(), 256, [0, 256])
        cdf  = hist.cumsum().astype(np.float64)
        cdf_min = cdf[cdf > 0].min()
        lut  = np.round((cdf - cdf_min) / (u8.size - cdf_min + 1e-8) * 255).astype(np.uint8)
        return to_float(lut[u8])

    def median_blur_on_roi(self, img_float, mask, ksize=5):
        u8      = to_uint8(img_float)
        blurred = cv2.medianBlur(u8, ksize)
        out     = img_float.copy()
        roi_bool = mask > 0.5
        out[roi_bool] = to_float(blurred)[roi_bool]
        return out

    def anisotropic_diffusion_norm(self, img_float, iters=5, kappa=30, gamma=0.10):
        """Perona-Malik: smooths homogeneous regions, preserves edges."""        img = img_float.copy().astype(np.float64)
        for _ in range(iters):
            dN = np.roll(img, -1, axis=0) - img
            dS = np.roll(img,  1, axis=0) - img
            dE = np.roll(img, -1, axis=1) - img
            dW = np.roll(img,  1, axis=1) - img
            cN = 1.0 / (1.0 + (dN / kappa)**2)
            cS = 1.0 / (1.0 + (dS / kappa)**2)
            cE = 1.0 / (1.0 + (dE / kappa)**2)
            cW = 1.0 / (1.0 + (dW / kappa)**2)
            img += gamma * (cN*dN + cS*dS + cE*dE + cW*dW)
        mn, mx = img.min(), img.max()
        return np.clip((img - mn) / (mx - mn + 1e-8), 0.0, 1.0).astype(np.float32)

    def __call__(self, pil_img):
        raw  = self.to_gray(pil_img)
        mask = LobarROIMask.compute(raw)
        img  = self.base_pipeline(pil_img)
        img  = LobarROIMask.apply_roi(img, mask)
        img  = self.bilateral_filter(img)
        img  = self.morphological_dilation(img, ksize=3)
        img  = self.histogram_matching(img)
        img  = self.median_blur_on_roi(img, mask, ksize=5)
        img  = self.anisotropic_diffusion_norm(img)
        img  = self.pblur(img)
        return self.to_rgb_pil(img)


bact_preprocessor  = BacterialPreprocessor()
viral_preprocessor = ViralPreprocessor()
base_preprocessor  = BaseChestPreprocessor()

print('✅ BaseChestPreprocessor  : GaussBlur → PercNorm → CLAHE')
print('✅ BacterialPreprocessor  : CLAHE → LobarROI → Sobel → Unsharp → HighPass → PeriBlur')
print('✅ ViralPreprocessor      : CLAHE → LobarROI → Bilateral → MorphDil → HistMatch → Median → DiffNorm → PeriBlur')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6: DENSITY INDEX (DI) — Auxiliary Pneumonia Classifier
#
#  DI = mean pixel intensity inside lung ROI /
#       std of local texture inside lung ROI + ε
#
#  High DI + low entropy  → dense consolidation  (BACTERIAL)
#  Mid  DI + high entropy → patchy/interstitial  (VIRAL)
#  High DI + high entropy → CO-INFECTION signal
#  Low  DI (<3)           → normal aeration      (NORMAL)
# ═══════════════════════════════════════════════════════════════════

def compute_density_index(pil_img, target_size=256):
    """
    Returns:
        di        (float)  — scalar density index
        di_class  (str)    — 'NORMAL' | 'VIRAL' | 'BACTERIAL' | 'CO-INFECTION'
        meta      (dict)   — mean, std, entropy
    """
    img = np.array(pil_img.convert('L').resize((target_size, target_size)))
    p2, p98 = np.percentile(img, (2, 98))
    img_n   = np.clip((img.astype(np.float32) - p2) / (p98 - p2 + 1e-8), 0, 1)

    blurred_raw = cv2.GaussianBlur(img, (5, 5), 0)
    mask  = LobarROIMask.compute(blurred_raw)
    lung  = img_n[mask > 0.5]
    if lung.size < 100:
        lung = img_n.flatten()

    mu    = float(np.mean(lung))
    sigma = float(np.std(lung))

    hist, _ = np.histogram(lung, bins=32, range=(0, 1))
    hist     = hist.astype(np.float32) + 1e-9
    hist    /= hist.sum()
    entropy  = float(-np.sum(hist * np.log2(hist)))

    di = mu / (sigma + 1e-8)

    if di < 3.0:
        di_class = 'NORMAL'
    elif di >= 5.5 and entropy < 3.5:
        di_class = 'BACTERIAL'
    elif 3.0 <= di < 5.5 and entropy >= 3.5:
        di_class = 'VIRAL'
    elif di >= 5.5 and entropy >= 3.5:
        di_class = 'CO-INFECTION'
    else:
        di_class = 'VIRAL'

    return di, di_class, dict(mean=mu, std=sigma, entropy=entropy)


def di_label_to_int(di_class):
    return {'NORMAL':0,'VIRAL':1,'BACTERIAL':2,'CO-INFECTION':3}.get(
            str(di_class).upper(), 4)

print('✅ compute_density_index() ready')
print('   DI = mean_lung / (std_texture + ε)')
print('   Thresholds: DI<3→NORMAL | DI≥5.5+low_H→BACT | 3≤DI<5.5+high_H→VIRAL')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7: TRANSFORM PIPELINES
#   Stage 1 (ViT)       — BaseChestPreprocessor
#   Stage 2 (Viral CNN) — ViralPreprocessor
#   Stage 3 (Bact  CNN) — BacterialPreprocessor
# ═══════════════════════════════════════════════════════════════════

class WithBasePreprocess:
    def __init__(self): self.p = BaseChestPreprocessor()
    def __call__(self, img): return self.p.to_rgb_pil(self.p.base_pipeline(img))

class WithViralPreprocess:
    def __init__(self): self.p = ViralPreprocessor()
    def __call__(self, img): return self.p(img)

class WithBactPreprocess:
    def __init__(self): self.p = BacterialPreprocessor()
    def __call__(self, img): return self.p(img)

# ── Stage 1: ViT ──────────────────────────────────────────────────
transform_train_vit = transforms.Compose([
    WithBasePreprocess(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_test_vit = transforms.Compose([
    WithBasePreprocess(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Stage 2: Viral CNN ────────────────────────────────────────────
transform_train_viral = transforms.Compose([
    WithViralPreprocess(),
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_test_viral = transforms.Compose([
    WithViralPreprocess(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Stage 3: Bacterial CNN ────────────────────────────────────────
transform_train_bact = transforms.Compose([
    WithBactPreprocess(),
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_test_bact = transforms.Compose([
    WithBactPreprocess(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('✅ Stage 1 transforms  — BasePreprocess + ViT augmentation')
print('✅ Stage 2 transforms  — ViralPreprocessor + CNN augmentation')
print('✅ Stage 3 transforms  — BacterialPreprocessor + CNN augmentation')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8: VISUALISE PREPROCESSING PIPELINES
# ═══════════════════════════════════════════════════════════════════

def visualise_preprocessing(img_path, figsize=(22, 9)):
    pil    = Image.open(img_path).convert('RGB')
    base_p = BaseChestPreprocessor()
    viral_p= ViralPreprocessor()
    bact_p = BacterialPreprocessor()

    orig         = np.array(pil)
    gray_b       = base_p.base_pipeline(pil)

    # Bacterial intermediates
    bact_sobel   = bact_p.sobel_enhance(gray_b)
    bact_unsharp = bact_p.unsharp_mask(bact_sobel)
    bact_hp      = bact_p.high_pass_filter(bact_unsharp)
    bact_final   = np.array(bact_p(pil).convert('L')) / 255.0

    # Viral intermediates
    vir_bilateral = viral_p.bilateral_filter(gray_b)
    vir_dilated   = viral_p.morphological_dilation(vir_bilateral)
    vir_hist      = viral_p.histogram_matching(vir_dilated)
    vir_diffnorm  = viral_p.anisotropic_diffusion_norm(vir_hist)
    vir_final     = np.array(viral_p(pil).convert('L')) / 255.0

    # Density index
    di, di_class, meta = compute_density_index(pil)

    fig, axes = plt.subplots(2, 7, figsize=figsize)
    fig.suptitle(
        f'Preprocessing Pipelines — DI={di:.2f} [{di_class}] | '
        f'mean={meta["mean"]:.3f}  std={meta["std"]:.3f}  entropy={meta["entropy"]:.2f}',
        fontsize=13, fontweight='bold')

    row0 = [orig, gray_b, bact_sobel, bact_unsharp, bact_hp, bact_final, None]
    row1 = [None, vir_bilateral, vir_dilated, vir_hist, vir_diffnorm, vir_final, None]
    t0   = ['Original','Base CLAHE','Bact: Sobel','Bact: Unsharp','Bact: HighPass','Bact: FINAL','']
    t1   = ['','Viral: Bilateral','Viral: MorphDil','Viral: HistMatch','Viral: DiffNorm','Viral: FINAL','']

    cmap = 'gray'
    for j, (img, title) in enumerate(zip(row0, t0)):
        axes[0,j].axis('off')
        if img is not None:
            axes[0,j].imshow(img if img.ndim==3 else img, cmap=None if img.ndim==3 else cmap)
        axes[0,j].set_title(title, fontsize=9)
    for j, (img, title) in enumerate(zip(row1, t1)):
        axes[1,j].axis('off')
        if img is not None:
            axes[1,j].imshow(img, cmap=cmap)
        axes[1,j].set_title(title, fontsize=9)

    plt.tight_layout()
    plt.savefig('/kaggle/working/preprocessing_viz.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'   Density Index = {di:.4f}  →  {di_class}')
    return di, di_class


# Run on a sample PNEUMONIA image
sample_imgs = (glob.glob(os.path.join(TRAIN_PATH, 'PNEUMONIA', '*.jpeg')) +
               glob.glob(os.path.join(TRAIN_PATH, 'PNEUMONIA', '*.jpg')))
if sample_imgs:
    _ = visualise_preprocessing(random.choice(sample_imgs))
else:
    print('⚠️  Run after paths are confirmed.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9: SHARED MODULES — CBAM + FOCAL LOSS
# ═══════════════════════════════════════════════════════════════════

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels, bias=False))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        a = self.fc(self.avg_pool(x).squeeze(-1).squeeze(-1))
        m = self.fc(self.max_pool(x).squeeze(-1).squeeze(-1))
        return self.sigmoid(a + m).unsqueeze(-1).unsqueeze(-1) * x


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1))) * x


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(in_channels, reduction)
        self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction

    def forward(self, inputs, targets):
        ce = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        fl = (1 - torch.exp(-ce)) ** self.gamma * ce
        if self.alpha is not None:
            fl = self.alpha[targets] * fl
        return fl.mean() if self.reduction == 'mean' else fl.sum()


class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=2.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma

    def forward(self, inputs, targets):
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        return torch.mean(self.alpha * (1 - torch.exp(-bce)) ** self.gamma * bce)


print('✅ CBAM + FocalLoss + BinaryFocalLoss ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10: MODEL DEFINITIONS
# ═══════════════════════════════════════════════════════════════════

class Stage1_ViT(nn.Module):
    """ViT-B/16: NORMAL(0) vs PNEUMONIA(1)"""    def __init__(self):
        super().__init__()
        self.vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        self.vit.heads.head = nn.Linear(self.vit.heads.head.in_features, 1)
    def forward(self, x): return self.vit(x)


class Stage2_PneumoFusionNet(nn.Module):
    """EfficientNet-B4 + DenseNet-121 + CBAM: NON-VIRAL(0) vs VIRAL(1)"""    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        e = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.eff = nn.Sequential(*list(e.children())[:-2])      # 1792
        d = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.den = d.features                                    # 1024
        self.ea  = CBAM(1792); self.da = CBAM(1024)
        fd = 512
        self.ep  = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(1792, fd), nn.BatchNorm1d(fd), nn.ReLU())
        self.dp  = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(1024, fd), nn.BatchNorm1d(fd), nn.ReLU())
        self.clf = nn.Sequential(nn.Linear(fd*2, 256), nn.BatchNorm1d(256),
                                  nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.clf(torch.cat([self.ep(self.ea(self.eff(x))),
                                   self.dp(self.da(self.den(x)))], dim=1))


class Stage3_BacterialFusionNet(nn.Module):
    """DenseNet-201 + EfficientNetV2-L + CBAM: NON-BACTERIAL(0) vs BACTERIAL(1)"""    def __init__(self, num_classes=2, dropout=0.35):
        super().__init__()
        d = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)
        self.den = d.features                                    # 1920
        self.dc  = CBAM(1920)
        e = models.efficientnet_v2_l(weights=models.EfficientNet_V2_L_Weights.IMAGENET1K_V1)
        self.eff = nn.Sequential(*list(e.children())[:-2])      # 1280
        self.ec  = CBAM(1280)
        fd = 512
        self.dp  = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(1920, fd), nn.BatchNorm1d(fd), nn.ReLU())
        self.ep  = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(1280, fd), nn.BatchNorm1d(fd), nn.ReLU())
        self.clf = nn.Sequential(nn.Linear(fd*2, 384), nn.BatchNorm1d(384),
                                  nn.GELU(), nn.Dropout(dropout), nn.Linear(384, num_classes))
    def forward(self, x):
        return self.clf(torch.cat([self.dp(self.dc(self.den(x))),
                                   self.ep(self.ec(self.eff(x)))], dim=1))


set_seed(42)
vit_model   = Stage1_ViT().to(device)
viral_model = Stage2_PneumoFusionNet().to(device)
bact_model  = Stage3_BacterialFusionNet().to(device)

print(f'✅ Stage 1  ViT-B/16           : {sum(p.numel() for p in vit_model.parameters())/1e6:.1f}M params')
print(f'✅ Stage 2  PneumoFusionNet    : {sum(p.numel() for p in viral_model.parameters())/1e6:.1f}M params')
print(f'✅ Stage 3  BacterialFusionNet : {sum(p.numel() for p in bact_model.parameters())/1e6:.1f}M params')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11: DATA PREPARATION — Split Pneumonia → Viral / Bacterial
#   Reads directly from the chest-xray-pneumonia ImageFolder dataset.
#   Filenames containing 'virus' → VIRAL class
#   Filenames containing 'bacteria' → BACTERIAL class
#   Normal images are shared across both Stage 2 and Stage 3.
#   Runs only once; skips if dirs already exist.
# ═══════════════════════════════════════════════════════════════════

def split_pneumonia_by_type(src_pneumonia, viral_dst, bact_dst, split=(0.8, 0.1, 0.1)):
    viral_files = sorted([f for f in glob.glob(
                          os.path.join(src_pneumonia,'**','*virus*'), recursive=True)
                          if f.lower().endswith(('.jpeg','.jpg','.png'))])
    bact_files  = sorted([f for f in glob.glob(
                          os.path.join(src_pneumonia,'**','*bacteria*'), recursive=True)
                          if f.lower().endswith(('.jpeg','.jpg','.png'))])
    normal_files = sorted(glob.glob(os.path.join(TRAIN_PATH,'NORMAL','*.jpeg')) +
                          glob.glob(os.path.join(TRAIN_PATH,'NORMAL','*.jpg')))

    def _split(files):
        n = len(files)
        t = int(n * split[0]); v = t + int(n * split[1])
        return files[:t], files[t:v], files[v:]

    vtr, vva, vte = _split(viral_files)
    btr, bva, bte = _split(bact_files)
    ntr, nva, nte = _split(normal_files)

    mapping = [
        (vtr, os.path.join(viral_dst, 'train', 'VIRAL')),
        (vva, os.path.join(viral_dst, 'val',   'VIRAL')),
        (vte, os.path.join(viral_dst, 'test',  'VIRAL')),
        (ntr, os.path.join(viral_dst, 'train', 'NORMAL')),
        (nva, os.path.join(viral_dst, 'val',   'NORMAL')),
        (nte, os.path.join(viral_dst, 'test',  'NORMAL')),
        (btr, os.path.join(bact_dst,  'train', 'BACTERIAL')),
        (bva, os.path.join(bact_dst,  'val',   'BACTERIAL')),
        (bte, os.path.join(bact_dst,  'test',  'BACTERIAL')),
        (ntr, os.path.join(bact_dst,  'train', 'NON_BACTERIAL')),
        (nva, os.path.join(bact_dst,  'val',   'NON_BACTERIAL')),
        (nte, os.path.join(bact_dst,  'test',  'NON_BACTERIAL')),
    ]
    for files, dst_dir in mapping:
        os.makedirs(dst_dir, exist_ok=True)
        for f in files:
            shutil.copy2(f, dst_dir)

    print(f'  Viral  → train:{len(vtr):,}  val:{len(vva):,}  test:{len(vte):,}')
    print(f'  Bact   → train:{len(btr):,}  val:{len(bva):,}  test:{len(bte):,}')
    print(f'  Normal → train:{len(ntr):,}  val:{len(nva):,}  test:{len(nte):,}')


if not os.path.exists(os.path.join(VIRAL_WORK, 'train')):
    print('🔧 Building viral/bacterial subdirectories...')
    split_pneumonia_by_type(
        src_pneumonia=os.path.join(TRAIN_PATH, 'PNEUMONIA'),
        viral_dst=VIRAL_WORK,
        bact_dst=BACT_WORK)
    print('✅ Directories built')
else:
    print('✅ Directories already exist — skipping rebuild')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12: DATASET CLASSES + DATALOADERS
#   Uses TRAIN_SUBSET / VAL_SUBSET / TEST_SUBSET from Cell 4.
#   All three stages respect the size controller.
# ═══════════════════════════════════════════════════════════════════

class SubsetImageFolder(Dataset):
    """
    ImageFolder wrapper that supports stratified subset sampling
    via sample_folder_dataset(). Compatible with weighted sampler.
    """
    def __init__(self, root, transform=None, subset_n=None, seed=42):
        base = ImageFolder(root, transform=None)
        if subset_n is not None and subset_n < len(base):
            sampled = sample_folder_dataset(base, subset_n, seed)
            self.image_paths = [s[0] for s in sampled]
            self.labels      = [s[1] for s in sampled]
        else:
            self.image_paths = [s[0] for s in base.samples]
            self.labels      = [s[1] for s in base.samples]
        self.transform = transform
        tag = f'(subset={subset_n})' if subset_n else '(full)'
        print(f'  ✅ {os.path.basename(root)} {tag}: {len(self.image_paths):,} images')

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)


class FolderDataset(Dataset):
    """
    Reads specific class subdirectories. Used for Stage 2 / 3
    where class names differ from the original ImageFolder.
    Supports subset sampling.
    """
    def __init__(self, root_dir, class_map, transform=None, subset_n=None, seed=42):
        self.transform = transform
        samples, labels_raw = [], []
        for cls, lbl in class_map.items():
            folder = os.path.join(root_dir, cls)
            if not os.path.exists(folder):
                continue
            for fname in sorted(os.listdir(folder)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    samples.append(os.path.join(folder, fname))
                    labels_raw.append(lbl)
        self.image_paths = samples
        self.labels      = labels_raw

        # Stratified subset
        if subset_n is not None and subset_n < len(samples):
            rng     = np.random.RandomState(seed)
            arr     = np.array(labels_raw)
            classes = np.unique(arr)
            sel     = []
            for cls in classes:
                idx = np.where(arr == cls)[0]
                k   = max(1, int(round((len(idx)/len(arr)) * subset_n)))
                k   = min(k, len(idx))
                sel.extend(rng.choice(idx, k, replace=False).tolist())
            rng.shuffle(sel)
            sel = sel[:subset_n]
            self.image_paths = [samples[i]    for i in sel]
            self.labels      = [labels_raw[i] for i in sel]

        tag = f'(subset={subset_n})' if subset_n else '(full)'
        print(f'  ✅ {os.path.basename(root_dir)} {tag}: {len(self.image_paths):,} images')

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)


def weighted_sampler(labels):
    cc = np.bincount(labels)
    w  = [1.0 / cc[l] for l in labels]
    return WeightedRandomSampler(w, len(w), replacement=True)


# ── Stage 1 loaders (ViT) ─────────────────────────────────────────
print("\nStage 1 (ViT — NORMAL vs PNEUMONIA):")
vit_train_ds = SubsetImageFolder(TRAIN_PATH, transform_train_vit, TRAIN_SUBSET)
vit_val_ds   = SubsetImageFolder(VAL_PATH,   transform_test_vit,  VAL_SUBSET)
vit_test_ds  = SubsetImageFolder(TEST_PATH,  transform_test_vit,  TEST_SUBSET)

vit_train_loader = DataLoader(vit_train_ds, batch_size=16,
                               sampler=weighted_sampler(vit_train_ds.labels),
                               num_workers=2, pin_memory=True)
vit_val_loader   = DataLoader(vit_val_ds,   batch_size=16, shuffle=False,
                               num_workers=2, pin_memory=True)
vit_test_loader  = DataLoader(vit_test_ds,  batch_size=16, shuffle=False,
                               num_workers=2, pin_memory=True)

# ── Stage 2 loaders (Viral CNN) ───────────────────────────────────
print("\nStage 2 (Viral CNN — NORMAL vs VIRAL):")
VMAP = {'NORMAL': 0, 'VIRAL': 1}
viral_train_ds = FolderDataset(f'{VIRAL_WORK}train/', VMAP, transform_train_viral, TRAIN_SUBSET)
viral_val_ds   = FolderDataset(f'{VIRAL_WORK}val/',   VMAP, transform_test_viral,  VAL_SUBSET)
viral_test_ds  = FolderDataset(f'{VIRAL_WORK}test/',  VMAP, transform_test_viral,  TEST_SUBSET)

viral_train_loader = DataLoader(viral_train_ds, batch_size=16,
                                 sampler=weighted_sampler(viral_train_ds.labels),
                                 num_workers=2, pin_memory=True)
viral_val_loader   = DataLoader(viral_val_ds,   batch_size=16, shuffle=False,
                                 num_workers=2, pin_memory=True)
viral_test_loader  = DataLoader(viral_test_ds,  batch_size=16, shuffle=False,
                                 num_workers=2, pin_memory=True)

# ── Stage 3 loaders (Bacterial CNN) ──────────────────────────────
print("\nStage 3 (Bacterial CNN — NON_BACTERIAL vs BACTERIAL):")
BMAP = {'NON_BACTERIAL': 0, 'BACTERIAL': 1}
bact_train_ds = FolderDataset(f'{BACT_WORK}train/', BMAP, transform_train_bact, TRAIN_SUBSET)
bact_val_ds   = FolderDataset(f'{BACT_WORK}val/',   BMAP, transform_test_bact,  VAL_SUBSET)
bact_test_ds  = FolderDataset(f'{BACT_WORK}test/',  BMAP, transform_test_bact,  TEST_SUBSET)

bact_train_loader = DataLoader(bact_train_ds, batch_size=16,
                                sampler=weighted_sampler(bact_train_ds.labels),
                                num_workers=2, pin_memory=True)
bact_val_loader   = DataLoader(bact_val_ds,   batch_size=16, shuffle=False,
                                num_workers=2, pin_memory=True)
bact_test_loader  = DataLoader(bact_test_ds,  batch_size=16, shuffle=False,
                                num_workers=2, pin_memory=True)

print(f"\nStage 1 → Train:{len(vit_train_ds):,}   Val:{len(vit_val_ds):,}   Test:{len(vit_test_ds):,}")
print(f"Stage 2 → Train:{len(viral_train_ds):,}   Val:{len(viral_val_ds):,}   Test:{len(viral_test_ds):,}")
print(f"Stage 3 → Train:{len(bact_train_ds):,}   Val:{len(bact_val_ds):,}   Test:{len(bact_test_ds):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 13: TRAINING & EVALUATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════

def train_one_epoch_vit(model, loader, opt, crit, sched):
    model.train(); total = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.float().unsqueeze(1).to(device)
        opt.zero_grad()
        loss = crit(model(imgs), lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item()
    sched.step()
    return total / len(loader)


def train_one_epoch_cnn(model, loader, opt, crit):
    model.train(); total = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        opt.zero_grad()
        loss = crit(model(imgs), lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); total += loss.item()
    return total / len(loader)


def evaluate_vit(model, loader):
    model.eval(); correct = n = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = (torch.sigmoid(model(imgs)) > 0.5).float().squeeze(1)
            correct += (preds == lbls.float()).sum().item(); n += lbls.size(0)
    return correct / n


def evaluate_cnn(model, loader):
    model.eval(); yt, yp, ys = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out   = model(imgs)
            probs = torch.softmax(out, dim=1)
            yt.extend(lbls.cpu().numpy())
            yp.extend(torch.argmax(out, dim=1).cpu().numpy())
            ys.extend(probs[:, 1].cpu().numpy())
    f1  = f1_score(yt, yp, zero_division=0)
    auc = roc_auc_score(yt, ys) if len(set(yt)) > 1 else 0.0
    acc = accuracy_score(yt, yp)
    return f1, auc, acc


print('✅ train_one_epoch_vit / train_one_epoch_cnn / evaluate_vit / evaluate_cnn ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 14: K-FOLD CROSS-VALIDATION FRAMEWORK
#   Stratified K-Fold on any of the three stages.
#   Best weights per fold kept; summary printed at end.
# ═══════════════════════════════════════════════════════════════════

class ListDataset(Dataset):
    """Generic dataset from a list of (path, label) tuples."""    def __init__(self, samples, transform=None):
        self.samples   = samples          # list of (path_str, int_label)
        self.labels    = [s[1] for s in samples]
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, lbl = self.samples[i]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(lbl, dtype=torch.long)


def make_model(stage: int):
    set_seed(42)
    if stage == 1:   return Stage1_ViT().to(device)
    elif stage == 2: return Stage2_PneumoFusionNet().to(device)
    else:            return Stage3_BacterialFusionNet().to(device)


def run_kfold_cv(samples, transform, stage: int,
                 n_splits=5, epochs=8,
                 ckpt_prefix='/kaggle/working/kfold', lr=3e-5):
    """
    Stratified K-Fold CV.

    Args:
        samples      : list of (path, label) tuples
        transform    : test-time transform (no aug for fair CV eval)
        stage        : 1, 2, or 3
        n_splits     : number of folds
        epochs       : epochs per fold
        ckpt_prefix  : checkpoint save prefix
        lr           : learning rate

    Returns:
        fold_results  : list of dicts
        best_weights  : state_dict from best fold
    """
    labels = np.array([s[1] for s in samples])
    skf    = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_results = []
    best_score   = -1.0
    best_weights = None

    print(f'\n🔁 K-Fold CV — Stage {stage} | {n_splits} folds × {epochs} epochs')
    print('═' * 65)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f'\n── Fold {fold+1}/{n_splits} ──────────────────────────────────────')
        tr_ds = ListDataset([samples[i] for i in tr_idx], transform)
        va_ds = ListDataset([samples[i] for i in va_idx], transform)

        fold_labels = labels[tr_idx]
        cc  = np.bincount(fold_labels)
        w   = [1.0 / cc[l] for l in fold_labels]
        sampler = WeightedRandomSampler(w, len(w), replacement=True)

        tr_loader = DataLoader(tr_ds, batch_size=16, sampler=sampler,
                               num_workers=2, pin_memory=True)
        va_loader = DataLoader(va_ds, batch_size=16, shuffle=False,
                               num_workers=2, pin_memory=True)

        model = make_model(stage)
        ckpt  = f'{ckpt_prefix}_s{stage}_fold{fold+1}.pth'
        best_fold_score = 0

        if stage == 1:
            crit  = BinaryFocalLoss(alpha=2.0, gamma=2.0)
            opt   = torch.optim.AdamW(model.parameters(), lr=lr or 2e-5, weight_decay=1e-2)
            sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=2)
            for ep in range(epochs):
                train_one_epoch_vit(model, tr_loader, opt, crit, sched)
                acc = evaluate_vit(model, va_loader)
                if acc > best_fold_score:
                    best_fold_score = acc
                    torch.save(model.state_dict(), ckpt)
            metric_name = 'Acc'
        else:
            alpha = (torch.tensor([1.0, 3.0]) if stage == 2
                     else torch.tensor([2.1, 1.0])).to(device)
            gamma = 2.0 if stage == 2 else 2.2
            crit  = FocalLoss(alpha=alpha, gamma=gamma)
            opt   = optim.AdamW(model.parameters(), lr=lr or 3e-5, weight_decay=0.01)
            sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
            for ep in range(epochs):
                train_one_epoch_cnn(model, tr_loader, opt, crit)
                sched.step()
                f1, auc, acc = evaluate_cnn(model, va_loader)
                score = 0.6 * f1 + 0.4 * auc
                if score > best_fold_score:
                    best_fold_score = score
                    torch.save(model.state_dict(), ckpt)
            model.load_state_dict(torch.load(ckpt, map_location=device))
            f1, auc, acc = evaluate_cnn(model, va_loader)
            best_fold_score = f1
            metric_name = 'F1'

        print(f'  Fold {fold+1} best {metric_name} = {best_fold_score:.4f}')
        fold_results.append(dict(fold=fold+1, metric=best_fold_score, ckpt=ckpt))

        if best_fold_score > best_score:
            best_score   = best_fold_score
            best_weights = torch.load(ckpt, map_location=device)

    scores = [r['metric'] for r in fold_results]
    print(f'\n📊 K-Fold Summary — Stage {stage}')
    print(f'   Mean : {np.mean(scores):.4f}  ±  {np.std(scores):.4f}')
    print(f'   Best : {np.max(scores):.4f}  (Fold {np.argmax(scores)+1})')
    print('═' * 65)
    return fold_results, best_weights


print('✅ ListDataset + run_kfold_cv() ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 15: TRAIN STAGE 1 — ViT-B/16  (+ K-Fold CV)
# ═══════════════════════════════════════════════════════════════════
set_seed(42)
print('🚀 STAGE 1: ViT-B/16  (NORMAL vs PNEUMONIA)')
print('=' * 65)

vit_crit  = BinaryFocalLoss(alpha=2.0, gamma=2.0)
vit_opt   = torch.optim.AdamW(vit_model.parameters(), lr=2e-5, weight_decay=1e-2)
vit_sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(vit_opt, T_0=5, T_mult=2)

best_vit = 0
for epoch in range(STAGE1_EPOCHS):
    loss    = train_one_epoch_vit(vit_model, vit_train_loader, vit_opt, vit_crit, vit_sched)
    val_acc = evaluate_vit(vit_model, vit_val_loader)
    lr_now  = vit_opt.param_groups[0]['lr']
    print(f'  E{epoch+1:2d}: Loss={loss:.4f} | Val={val_acc:.1%} | LR={lr_now:.2e}')
    if val_acc > best_vit:
        best_vit = val_acc
        torch.save({'model_state_dict': vit_model.state_dict(), 'val_acc': val_acc}, CKPT_VIT)
        print(f'  ⭐ SAVED  {val_acc:.1%}')

print(f'\n🏆 Stage 1 Best Val Acc: {best_vit:.1%}')

# ── K-Fold CV ─────────────────────────────────────────────────────
print('\n📂 Running K-Fold CV on Stage 1...')
s1_all_samples = list(zip(vit_train_ds.image_paths, vit_train_ds.labels))
s1_kfold_results, s1_best_weights = run_kfold_cv(
    samples=s1_all_samples,
    transform=transform_test_vit,
    stage=1, n_splits=N_FOLDS, epochs=KFOLD_EPOCHS,
    ckpt_prefix='/kaggle/working/kfold', lr=2e-5)

if s1_best_weights is not None:
    vit_model.load_state_dict(s1_best_weights)
    print('✅ Best K-Fold weights loaded into vit_model')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 16: TRAIN STAGE 2 — PneumoFusionNet  (Viral)  [+ K-Fold CV]
# ═══════════════════════════════════════════════════════════════════
set_seed(42)
print('🚀 STAGE 2: PneumoFusionNet  (NON-VIRAL vs VIRAL)')
print('=' * 65)

v_crit  = FocalLoss(alpha=torch.tensor([1.0, 3.0]).to(device), gamma=2.0)
v_opt   = optim.AdamW(viral_model.parameters(), lr=3e-5, weight_decay=0.01)
v_sched = optim.lr_scheduler.CosineAnnealingLR(v_opt, T_max=STAGE2_EPOCHS, eta_min=1e-6)

best_viral = 0
for epoch in range(STAGE2_EPOCHS):
    loss                     = train_one_epoch_cnn(viral_model, viral_train_loader, v_opt, v_crit)
    val_f1, val_auc, val_acc = evaluate_cnn(viral_model, viral_val_loader)
    v_sched.step(); lr_now   = v_opt.param_groups[0]['lr']
    print(f'  E{epoch+1:2d}: Loss={loss:.4f} | F1={val_f1:.4f} | AUC={val_auc:.4f} | Acc={val_acc:.3f} | LR={lr_now:.2e}')
    if val_f1 > best_viral:
        best_viral = val_f1
        torch.save({'model_state_dict': viral_model.state_dict(), 'val_f1': val_f1}, CKPT_VIRAL)
        print(f'  ⭐ SAVED  F1={val_f1:.4f}')

print(f'\n🏆 Stage 2 Best Val F1: {best_viral:.4f}')

# ── K-Fold CV ─────────────────────────────────────────────────────
print('\n📂 Running K-Fold CV on Stage 2 (Viral)...')
s2_all_samples = (list(zip(viral_train_ds.image_paths, viral_train_ds.labels)) +
                  list(zip(viral_val_ds.image_paths,   viral_val_ds.labels)))
s2_kfold_results, s2_best_weights = run_kfold_cv(
    samples=s2_all_samples,
    transform=transform_test_viral,
    stage=2, n_splits=N_FOLDS, epochs=KFOLD_EPOCHS,
    ckpt_prefix='/kaggle/working/kfold', lr=3e-5)

if s2_best_weights is not None:
    viral_model.load_state_dict(s2_best_weights)
    print('✅ Best K-Fold weights loaded into viral_model')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 17: TRAIN STAGE 3 — BacterialFusionNet  [+ K-Fold CV]
# ═══════════════════════════════════════════════════════════════════
set_seed(42)
print('🚀 STAGE 3: BacterialFusionNet  (NON-BACTERIAL vs BACTERIAL)')
print('=' * 65)

b_crit  = FocalLoss(alpha=torch.tensor([2.1, 1.0]).to(device), gamma=2.2)
b_opt   = optim.AdamW(bact_model.parameters(), lr=5e-5, weight_decay=0.01)
b_sched = optim.lr_scheduler.CosineAnnealingLR(b_opt, T_max=STAGE3_EPOCHS, eta_min=1e-6)

best_bact = 0
for epoch in range(STAGE3_EPOCHS):
    loss                     = train_one_epoch_cnn(bact_model, bact_train_loader, b_opt, b_crit)
    val_f1, val_auc, val_acc = evaluate_cnn(bact_model, bact_val_loader)
    b_sched.step(); lr_now   = b_opt.param_groups[0]['lr']
    print(f'  E{epoch+1:2d}: Loss={loss:.4f} | F1={val_f1:.4f} | AUC={val_auc:.4f} | Acc={val_acc:.3f} | LR={lr_now:.2e}')
    if val_f1 > best_bact:
        best_bact = val_f1
        torch.save({'model_state_dict': bact_model.state_dict(), 'val_f1': val_f1}, CKPT_BACT)
        print(f'  ⭐ SAVED  F1={val_f1:.4f}')

print(f'\n🏆 Stage 3 Best Val F1: {best_bact:.4f}')

# ── K-Fold CV ─────────────────────────────────────────────────────
print('\n📂 Running K-Fold CV on Stage 3 (Bacterial)...')
s3_all_samples = (list(zip(bact_train_ds.image_paths, bact_train_ds.labels)) +
                  list(zip(bact_val_ds.image_paths,   bact_val_ds.labels)))
s3_kfold_results, s3_best_weights = run_kfold_cv(
    samples=s3_all_samples,
    transform=transform_test_bact,
    stage=3, n_splits=N_FOLDS, epochs=KFOLD_EPOCHS,
    ckpt_prefix='/kaggle/working/kfold', lr=5e-5)

if s3_best_weights is not None:
    bact_model.load_state_dict(s3_best_weights)
    print('✅ Best K-Fold weights loaded into bact_model')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 18: LOAD BEST CHECKPOINTS
#   Run this cell to reload the best saved weights if you want to
#   override the K-Fold weights with the single-run best checkpoint.
# ═══════════════════════════════════════════════════════════════════

def load_ckpt(model, path, key='model_state_dict'):
    if os.path.exists(path):
        ck = torch.load(path, map_location=device)
        model.load_state_dict(ck[key] if key in ck else ck)
        print(f'  ✅ Loaded  {path}')
    else:
        print(f'  ⚠️  Not found: {path}')
    model.eval()

load_ckpt(vit_model,   CKPT_VIT)
load_ckpt(viral_model, CKPT_VIRAL)
load_ckpt(bact_model,  CKPT_BACT)
print('All models in eval() mode.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 19: PARALLEL CASCADE PIPELINE — CORE  (with Density Index)
#
#  Decision flow per image:
#   Stage 1 (ViT)          → NORMAL / PNEUMONIA
#   If PNEUMONIA (parallel):
#     Stage 2 (Viral CNN)  → p_viral
#     Stage 3 (Bact  CNN)  → p_bacterial
#     Density Index (DI)   → di_class
#   Soft fusion: p_fused = 0.75×CNN + 0.25×DI
#   Labels:
#     0=NORMAL | 1=VIRAL | 2=BACTERIAL | 3=CO-INFECTION | 4=MISCELLANEOUS
# ═══════════════════════════════════════════════════════════════════

LABEL_MAP  = {0:'NORMAL', 1:'VIRAL PNEUMONIA',
              2:'BACTERIAL PNEUMONIA', 3:'CO-INFECTION', 4:'MISCELLANEOUS'}
DI_WEIGHT  = 0.25
CNN_WEIGHT = 1.0 - DI_WEIGHT


def cascade_predict(iv, ic_v, ic_b, raw_pil=None):
    """
    Args:
        iv      : (1,3,224,224) ViT-preprocessed tensor
        ic_v    : (1,3,224,224) viral-preprocessed tensor
        ic_b    : (1,3,224,224) bact-preprocessed tensor
        raw_pil : original PIL image for Density Index (optional)
    Returns dict with label, label_name, attempt, probabilities, DI info.
    """
    iv  = iv.to(device)
    ic_v = ic_v.to(device)
    ic_b = ic_b.to(device)

    di, di_class, di_meta = (None, 'UNKNOWN', {})
    if raw_pil is not None:
        di, di_class, di_meta = compute_density_index(raw_pil)

    pv = pb = 0.0
    for attempt in range(1, 3):
        with torch.no_grad():
            p_pneu = torch.sigmoid(vit_model(iv)).item()

        if p_pneu <= 0.5:
            return dict(label=0, label_name='NORMAL', attempt=attempt,
                        p_pneumonia=p_pneu, p_viral=0.0, p_bacterial=0.0,
                        density_index=di, di_class=di_class)

        with torch.no_grad():
            p_viral = torch.softmax(viral_model(ic_v), dim=1).squeeze()[1].item()
            p_bact  = torch.softmax(bact_model(ic_b),  dim=1).squeeze()[1].item()

        if di is not None:
            di_v = 1.0 if di_class in ['VIRAL',     'CO-INFECTION'] else 0.0
            di_b = 1.0 if di_class in ['BACTERIAL', 'CO-INFECTION'] else 0.0
            pv   = CNN_WEIGHT * p_viral + DI_WEIGHT * di_v
            pb   = CNN_WEIGHT * p_bact  + DI_WEIGHT * di_b
        else:
            pv, pb = p_viral, p_bact

        if pv > 0.5 and pb > 0.5:
            return dict(label=3, label_name='CO-INFECTION',       attempt=attempt,
                        p_pneumonia=p_pneu, p_viral=pv, p_bacterial=pb,
                        density_index=di, di_class=di_class)
        if pv > 0.5:
            return dict(label=1, label_name='VIRAL PNEUMONIA',    attempt=attempt,
                        p_pneumonia=p_pneu, p_viral=pv, p_bacterial=pb,
                        density_index=di, di_class=di_class)
        if pb > 0.5:
            return dict(label=2, label_name='BACTERIAL PNEUMONIA',attempt=attempt,
                        p_pneumonia=p_pneu, p_viral=pv, p_bacterial=pb,
                        density_index=di, di_class=di_class)

        if attempt == 1:
            continue  # retry

    return dict(label=4, label_name='MISCELLANEOUS', attempt=2,
                p_pneumonia=p_pneu, p_viral=pv, p_bacterial=pb,
                density_index=di, di_class=di_class)


print('✅ cascade_predict() — Parallel Cascade + Density Index Fusion')
print(f'   DI weight = {DI_WEIGHT:.0%}  |  CNN weight = {CNN_WEIGHT:.0%}')
print('   0=NORMAL | 1=VIRAL | 2=BACTERIAL | 3=CO-INFECTION | 4=MISCELLANEOUS')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 20: RUN CASCADE ON FULL TEST SET
# ═══════════════════════════════════════════════════════════════════

class DualTransformDataset(Dataset):
    """
    Returns (img_vit, img_viral, img_bact, raw_pil, true_label, filename)
    for every image in the given ImageFolder root.
    Supports subset sampling via TEST_SUBSET.
    """
    def __init__(self, root_path, t_vit, t_viral, t_bact, subset_n=None, seed=42):
        base = ImageFolder(root_path)
        if subset_n is not None and subset_n < len(base):
            sampled = sample_folder_dataset(base, subset_n, seed)
        else:
            sampled = base.samples
        self.samples = sampled
        self.labels  = [s[1] for s in sampled]
        self.t_vit   = t_vit
        self.t_viral = t_viral
        self.t_bact  = t_bact
        tag = f'(subset={subset_n})' if subset_n else '(full)'
        print(f'  ✅ DualTransformDataset {tag}: {len(self.samples):,} images')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return (self.t_vit(img), self.t_viral(img), self.t_bact(img),
                img, lbl, os.path.basename(path))


print('Building test dataset...')
dual_test = DualTransformDataset(
    TEST_PATH,
    transform_test_vit, transform_test_viral, transform_test_bact,
    subset_n=TEST_SUBSET)

print(f'\n🔍 Running cascade on {len(dual_test):,} test images...')
results = []
for idx in tqdm(range(len(dual_test)), desc='Cascade inference'):
    iv, ic_v, ic_b, raw_pil, true_lbl, fname = dual_test[idx]
    res = cascade_predict(
        iv.unsqueeze(0), ic_v.unsqueeze(0), ic_b.unsqueeze(0), raw_pil)

    fn_lower = fname.lower()
    if true_lbl == 0:         subtype = 'NORMAL'
    elif 'virus'    in fn_lower: subtype = 'VIRAL'
    elif 'bacteria' in fn_lower: subtype = 'BACTERIAL'
    else:                        subtype = 'PNEUMONIA'  # generic label

    res['true_label']   = true_lbl
    res['true_subtype'] = subtype
    res['filename']     = fname
    results.append(res)

df = pd.DataFrame(results)
print('\n✅ Cascade inference complete')
print('\n📊 PREDICTED LABEL DISTRIBUTION')
print('=' * 50)
for lbl, name in LABEL_MAP.items():
    n = int((df['label'] == lbl).sum())
    print(f'   {name:<25}: {n:4d}  ({n/len(df)*100:.1f}%)')
print(f'\n   Retry triggered : {int((df["attempt"] > 1).sum())}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 21: FULL ACCURACY METRICS
# ═══════════════════════════════════════════════════════════════════

print('\n' + '═'*65)
print('  FULL PIPELINE ACCURACY REPORT')
print('═'*65)

# Stage 1: Normal vs Pneumonia
true_bin = df['true_label'].values
pred_bin = (df['label'] != 0).astype(int).values

s1_acc = accuracy_score(true_bin, pred_bin)
s1_f1  = f1_score(true_bin, pred_bin)
s1_pre = precision_score(true_bin, pred_bin)
s1_rec = recall_score(true_bin, pred_bin)

print('\n▶ Stage 1 — NORMAL vs PNEUMONIA')
print(f'   Accuracy  : {s1_acc:.4f}  ({s1_acc:.2%})')
print(f'   Precision : {s1_pre:.4f}')
print(f'   Recall    : {s1_rec:.4f}')
print(f'   F1        : {s1_f1:.4f}')
print(classification_report(true_bin, pred_bin,
      target_names=['NORMAL','PNEUMONIA'], digits=4))

# Subtype accuracy
sub_df = df[df['true_subtype'].isin(['VIRAL','BACTERIAL'])].copy()

def subtype_hit(row):
    l, ts = row['label'], row['true_subtype']
    if ts == 'VIRAL':     return int(l in [1, 3])
    if ts == 'BACTERIAL': return int(l in [2, 3])
    return 0

sub_df['hit'] = sub_df.apply(subtype_hit, axis=1)
viral_sub     = sub_df[sub_df['true_subtype'] == 'VIRAL']
bact_sub      = sub_df[sub_df['true_subtype'] == 'BACTERIAL']
subtype_acc   = sub_df['hit'].mean()
viral_acc     = viral_sub['hit'].mean() if len(viral_sub) > 0 else 0.0
bact_acc      = bact_sub['hit'].mean()  if len(bact_sub)  > 0 else 0.0

print('\n▶ Stage 2+3 — Subtype Classification')
print(f'   Viral Acc           : {viral_acc:.4f}  ({viral_acc:.2%})  [n={len(viral_sub)}]')
print(f'   Bacterial Acc       : {bact_acc:.4f}  ({bact_acc:.2%})  [n={len(bact_sub)}]')
print(f'   Overall Subtype Acc : {subtype_acc:.4f}  ({subtype_acc:.2%})')

# End-to-end
def overall_hit(row):
    l, ts = row['label'], row['true_subtype']
    if ts == 'NORMAL':    return int(l == 0)
    if ts == 'VIRAL':     return int(l in [1, 3])
    if ts == 'BACTERIAL': return int(l in [2, 3])
    return 0

df['overall_hit'] = df.apply(overall_hit, axis=1)
overall_acc = df['overall_hit'].mean()
co_count    = int((df['label'] == 3).sum())
misc_count  = int((df['label'] == 4).sum())
retry_n     = int((df['attempt'] > 1).sum())

# Density Index accuracy
di_pred_int = df['di_class'].apply(di_label_to_int)
di_true_int = df['true_subtype'].apply(
    lambda s: {'NORMAL':0,'VIRAL':1,'BACTERIAL':2}.get(s.upper(), 4))
di_mask = di_true_int != 4
di_acc  = accuracy_score(di_true_int[di_mask], di_pred_int[di_mask]) if di_mask.sum() > 0 else 0.0

print(f'\n▶ Density Index Standalone Acc : {di_acc:.4f}  ({di_acc:.2%})')

# Final box
print('\n' + '╔' + '═'*62 + '╗')
print('║   📊 PIPELINE RESULTS SUMMARY' + ' '*32 + '║')
print('╠' + '═'*62 + '╣')
print(f'║  Train={TRAIN_SUBSET or "FULL"!s:<5}  Val={VAL_SUBSET or "FULL"!s:<4}  '
      f'Test={TEST_SUBSET or "FULL"!s:<4}  Folds={N_FOLDS}             ║')
print('╠' + '═'*62 + '╣')
print(f'║  Stage 1  Accuracy (NORMAL vs PNEUMONIA) : {s1_acc:>7.2%}       ║')
print(f'║  Stage 1  F1                             : {s1_f1:>7.4f}       ║')
print(f'║  Stage 1  Precision                      : {s1_pre:>7.4f}       ║')
print(f'║  Stage 1  Recall                         : {s1_rec:>7.4f}       ║')
print('╠' + '═'*62 + '╣')
print(f'║  Viral subtype Accuracy                  : {viral_acc:>7.2%}       ║')
print(f'║  Bacterial subtype Accuracy              : {bact_acc:>7.2%}       ║')
print(f'║  Overall Subtype Accuracy                : {subtype_acc:>7.2%}       ║')
print('╠' + '═'*62 + '╣')
print(f'║  Density Index Standalone Accuracy       : {di_acc:>7.2%}       ║')
print('╠' + '═'*62 + '╣')
print(f'║  ✅ END-TO-END ACCURACY                  : {overall_acc:>7.2%}       ║')
print('╠' + '═'*62 + '╣')
print(f'║  NORMAL              : {int((df.label==0).sum()):>4}   CO-INFECTION : {co_count:>4}           ║')
print(f'║  VIRAL PNEUMONIA     : {int((df.label==1).sum()):>4}   MISCELLANEOUS: {misc_count:>4}           ║')
print(f'║  BACTERIAL PNEUMONIA : {int((df.label==2).sum()):>4}   Retry count  : {retry_n:>4}           ║')
print('╠' + '═'*62 + '╣')
s2sc = [r['metric'] for r in s2_kfold_results]
s3sc = [r['metric'] for r in s3_kfold_results]
s1sc = [r['metric'] for r in s1_kfold_results]
print(f'║  K-Fold S1 Acc   mean={np.mean(s1sc):.4f} ± {np.std(s1sc):.4f}              ║')
print(f'║  K-Fold S2 F1    mean={np.mean(s2sc):.4f} ± {np.std(s2sc):.4f}              ║')
print(f'║  K-Fold S3 F1    mean={np.mean(s3sc):.4f} ± {np.std(s3sc):.4f}              ║')
print('╚' + '═'*62 + '╝')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 22: VISUALISATIONS  (3×3 grid)
# ═══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 3, figsize=(20, 18))
fig.suptitle('Enhanced Cascade Pipeline v4 — Test Set Results',
             fontsize=16, fontweight='bold')

# 1) Label distribution
ax = axes[0,0]
names  = [LABEL_MAP[i] for i in range(5)]
counts = [int((df['label']==i).sum()) for i in range(5)]
colors = ['#4CAF50','#2196F3','#FF9800','#9C27B0','#9E9E9E']
bars   = ax.bar(names, counts, color=colors, edgecolor='white')
ax.set_title('Final Label Distribution', fontweight='bold')
ax.set_ylabel('Count'); ax.tick_params(axis='x', rotation=18)
for bar, c in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            str(c), ha='center', fontweight='bold')

# 2) Stage 1 confusion matrix
ax = axes[0,1]
cm = confusion_matrix(true_bin, pred_bin)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['NORMAL','PNEUMONIA'], yticklabels=['NORMAL','PNEUMONIA'])
ax.set_title(f'Stage 1 Confusion Matrix  (Acc={s1_acc:.2%})', fontweight='bold')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')

# 3) Accuracy by category
ax = axes[0,2]
cats  = ['Stage 1', 'Viral', 'Bacterial', 'Overall']
accs  = [s1_acc, viral_acc, bact_acc, overall_acc]
bars2 = ax.bar(cats, [a*100 for a in accs],
               color=['#607D8B','#2196F3','#FF9800','#4CAF50'])
ax.set_title('Accuracy by Category', fontweight='bold')
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 115)
for bar, a in zip(bars2, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{a:.1%}', ha='center', fontweight='bold', fontsize=9)

# 4) K-Fold CV per fold (Stage 2 & 3)
ax = axes[1,0]
s2_scores = [r['metric'] for r in s2_kfold_results]
s3_scores = [r['metric'] for r in s3_kfold_results]
x = np.arange(len(s2_scores))
ax.bar(x-0.2, s2_scores, 0.35, label='Stage2 Viral F1',  color='#2196F3')
ax.bar(x+0.2, s3_scores, 0.35, label='Stage3 Bact  F1',  color='#FF9800')
ax.axhline(np.mean(s2_scores), color='#1565C0', ls='--', lw=1.5)
ax.axhline(np.mean(s3_scores), color='#E65100', ls='--', lw=1.5)
ax.set_xticks(x); ax.set_xticklabels([f'Fold {i+1}' for i in x])
ax.set_ylabel('F1 Score')
ax.set_title('K-Fold CV Results — Stage 2 & 3', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1.1)

# 5) Density Index distribution
ax = axes[1,1]
di_vals   = df['density_index'].dropna()
di_labels = df.loc[di_vals.index, 'true_subtype']
for lbl, col in [('NORMAL','#4CAF50'),('VIRAL','#2196F3'),('BACTERIAL','#FF9800')]:
    subset = di_vals[di_labels == lbl]
    if len(subset):
        ax.hist(subset, bins=25, alpha=0.6, label=lbl, color=col, edgecolor='white')
ax.axvline(3.0, color='gray',  ls='--', lw=1, label='DI=3  (NORMAL)')
ax.axvline(5.5, color='black', ls='--', lw=1, label='DI=5.5 (BACT)')
ax.set_xlabel('Density Index'); ax.set_ylabel('Count')
ax.set_title('Density Index Distribution by Class', fontweight='bold')
ax.legend(fontsize=8)

# 6) DI agreement pie
ax = axes[1,2]
di_agree = (df['di_class'].str.upper() == df['true_subtype'].str.upper())
ax.pie([di_agree.sum(), (~di_agree).sum()],
       labels=['DI Agrees', 'DI Disagrees'],
       colors=['#4CAF50','#EF5350'],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Density Index Agreement', fontweight='bold')

# 7) Retry / Misc pie
ax = axes[2,0]
classified_retry = retry_n - misc_count
ax.pie([len(df)-retry_n, classified_retry, misc_count],
       labels=['1st-pass classified','Retry → classified','MISCELLANEOUS'],
       colors=['#4CAF50','#FFC107','#F44336'],
       autopct='%1.1f%%', startangle=140, textprops={'fontsize':10})
ax.set_title('Retry Counter Analysis', fontweight='bold')

# 8) p_viral vs p_bacterial scatter
ax = axes[2,1]
cmap_lbl = {0:'#4CAF50',1:'#2196F3',2:'#FF9800',3:'#9C27B0',4:'#9E9E9E'}
for lbl in range(5):
    sub = df[df['label']==lbl]
    if len(sub):
        ax.scatter(sub['p_viral'], sub['p_bacterial'],
                   c=cmap_lbl[lbl], label=LABEL_MAP[lbl], alpha=0.5, s=20)
ax.axhline(0.5, color='gray', ls='--', lw=1)
ax.axvline(0.5, color='gray', ls='--', lw=1)
ax.set_xlabel('p_viral (fused)'); ax.set_ylabel('p_bacterial (fused)')
ax.set_title('Viral vs Bacterial Probability Scatter', fontweight='bold')
ax.legend(fontsize=7, markerscale=2)

# 9) K-Fold Stage 1
ax = axes[2,2]
s1_scores = [r['metric'] for r in s1_kfold_results]
ax.bar(range(1, len(s1_scores)+1), s1_scores, color='#607D8B')
ax.axhline(np.mean(s1_scores), color='red', ls='--', lw=1.5,
           label=f'Mean={np.mean(s1_scores):.4f}')
ax.set_xlabel('Fold'); ax.set_ylabel('Accuracy')
ax.set_title('Stage 1 K-Fold Accuracy', fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/cascade_v4_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → /kaggle/working/cascade_v4_results.png')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 23: SINGLE IMAGE DEMO
# ═══════════════════════════════════════════════════════════════════

def classify_xray(image_path, verbose=True):
    """Classify one chest X-ray through the full cascade pipeline."""    raw_pil = Image.open(image_path).convert('RGB')
    iv  = transform_test_vit(raw_pil).unsqueeze(0)
    ic_v = transform_test_viral(raw_pil).unsqueeze(0)
    ic_b = transform_test_bact(raw_pil).unsqueeze(0)

    di, di_class, di_meta = compute_density_index(raw_pil)
    if verbose:
        print(f'\n📐 Density Index  = {di:.4f}  →  {di_class}')
        print(f'   (mean={di_meta["mean"]:.3f}  std={di_meta["std"]:.3f}  '
              f'entropy={di_meta["entropy"]:.2f})')

    res = cascade_predict(iv, ic_v, ic_b, raw_pil)

    if verbose:
        print(f'\n  ══ FINAL: {res["label_name"]}  |  DI={di:.2f} [{di_class}]')
        print(f'     p_viral={res["p_viral"]:.3f}  '
              f'p_bact={res["p_bacterial"]:.3f}  '
              f'p_pneumonia={res["p_pneumonia"]:.3f}  '
              f'attempts={res["attempt"]} ══')
    return res


# Pick a random test image
demo_imgs = (glob.glob(os.path.join(TEST_PATH, '**', '*.jpeg'), recursive=True) +
             glob.glob(os.path.join(TEST_PATH, '**', '*.jpg'),  recursive=True))
if demo_imgs:
    demo_path = random.choice(demo_imgs)
    print(f'Demo image: {demo_path}')
    _ = classify_xray(demo_path, verbose=True)
else:
    print('⚠️  No test images found — check TEST_PATH.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 24: 💾 SAVE MODELS — PERSISTENT STORAGE
#
#  Saves all three model weights to /kaggle/working/saved_models/
#  so they survive between Kaggle sessions.
#
#  HOW TO RELOAD NEXT SESSION:
#    1. Go to Notebook → Output tab → click "Add to Dataset"
#    2. Next session: add that dataset as an Input
#    3. Set SAVED_MODELS_DIR to the input path and run load_ckpt()
# ═══════════════════════════════════════════════════════════════════

import shutil

# ── Save individual stage checkpoints ────────────────────────────
def save_all_models():
    bundle = {
        'stage1_vit'      : vit_model.state_dict(),
        'stage2_viral'    : viral_model.state_dict(),
        'stage3_bact'     : bact_model.state_dict(),
        'kfold_s1_results': s1_kfold_results,
        'kfold_s2_results': s2_kfold_results,
        'kfold_s3_results': s3_kfold_results,
        'hyperparams': {
            'TRAIN_SUBSET': TRAIN_SUBSET,
            'VAL_SUBSET'  : VAL_SUBSET,
            'TEST_SUBSET' : TEST_SUBSET,
            'STAGE1_EPOCHS': STAGE1_EPOCHS,
            'STAGE2_EPOCHS': STAGE2_EPOCHS,
            'STAGE3_EPOCHS': STAGE3_EPOCHS,
            'N_FOLDS'      : N_FOLDS,
        }
    }
    bundle_path = os.path.join(SAVED_MODELS_DIR, 'cascade_v4_bundle.pth')
    torch.save(bundle, bundle_path)
    print(f'✅ Bundle saved → {bundle_path}')

    # Also copy the individual best-checkpoint .pth files
    for src_path in [CKPT_VIT, CKPT_VIRAL, CKPT_BACT]:
        if os.path.exists(src_path):
            dst = os.path.join(SAVED_MODELS_DIR, os.path.basename(src_path))
            shutil.copy2(src_path, dst)
            print(f'✅ Copied  {os.path.basename(src_path)}  → saved_models/')

    # Save results CSV
    df.to_csv(os.path.join(SAVED_MODELS_DIR, 'cascade_v4_results.csv'), index=False)
    print('✅ cascade_v4_results.csv  saved')

save_all_models()

print('\n╔══════════════════════════════════════════════════════════════╗')
print('║   ENHANCED CASCADE PIPELINE v4 — COMPLETE                   ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Train={str(TRAIN_SUBSET or "FULL"):<5}  Val={str(VAL_SUBSET or "FULL"):<4}  '
      f'Test={str(TEST_SUBSET or "FULL"):<4}  Folds={N_FOLDS}              ║')
print(f'║  Epochs  S1={STAGE1_EPOCHS:<3}  S2={STAGE2_EPOCHS:<3}  S3={STAGE3_EPOCHS:<3}  '
      f'KFold={KFOLD_EPOCHS}×{N_FOLDS}           ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Stage 1  Accuracy  : {s1_acc:>7.2%}                         ║')
print(f'║  Stage 1  F1        : {s1_f1:>7.4f}                         ║')
print(f'║  Viral Accuracy     : {viral_acc:>7.2%}                         ║')
print(f'║  Bacterial Accuracy : {bact_acc:>7.2%}                         ║')
print(f'║  Subtype Accuracy   : {subtype_acc:>7.2%}                         ║')
print(f'║  ✅ END-TO-END ACC  : {overall_acc:>7.2%}                         ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  Files saved to /kaggle/working/saved_models/               ║')
print('║  → cascade_v4_bundle.pth  (all weights + kfold + config)   ║')
print('║  → stage1_vit_best.pth                                      ║')
print('║  → stage2_viral_best.pth                                     ║')
print('║  → stage3_bact_best.pth                                      ║')
print('║  → cascade_v4_results.csv                                    ║')
print('║                                                              ║')
print('║  ⚡ TO PERSIST: Notebook → Output → Add to Dataset           ║')
print('╚══════════════════════════════════════════════════════════════╝')